# **AI ASSIGNMENT # 02**
## **I242533**
## **Dawar Ahmed** 
## **DS - *B***

## GitHub Repository
[View the complete project on GitHub](https://github.com/DawarAhmed27/i242533_DawarAhmed_A2/blob/main/i242533_DawarAhmed_A2.ipynb)

In [7]:
import random
import tkinter as tk
from tkinter import scrolledtext, messagebox
import threading
import time
import queue

#CARD CLASS
class Card:
    def __init__(self, color, value):
        self.color = color
        self.value = str(value)

    def __repr__(self):
        return f"{self.color} {self.value}"

    def matches(self, other_card):
        return self.color == other_card.color or self.value == other_card.value

#DECK GENERATOR
def generate_deck():
    colors = ['Red', 'Blue', 'Green', 'Yellow']
    numbers = list(range(10))
    deck = []
    for color in colors:
        for num in numbers:
            deck.append(Card(color, num))
        deck.append(Card(color, 'Skip'))  # 1 skip per color
    random.shuffle(deck)
    return deck

#LEGAL MOVE GENERATOR
def get_valid_moves(hand, top_card):
    return [card for card in hand if card.matches(top_card)]

#GAME STATE
class GameState:
    def __init__(self, ai_cards, opponent1_card_count, opponent2_card_count, top_card, deck):
        self.ai_cards = ai_cards
        self.opponent1_cards = opponent1_card_count
        self.opponent2_cards = opponent2_card_count
        self.top_card = top_card
        self.deck = deck if isinstance(deck, list) else []

    def __repr__(self):
        return (f"Top: {self.top_card} | AI:{len(self.ai_cards)} "
                f"| Opp1:{self.opponent1_cards} | Opp2:{self.opponent2_cards} "
                f"| Deck:{len(self.deck)}")

def evaluate_state(state, strategy="Defensive"):
    c_ai  = len(state.ai_cards)
    c_opp = (state.opponent1_cards + state.opponent2_cards) / 2.0
    s_count = sum(1 for card in state.ai_cards if card.value == 'Skip')

    score = 50 - (5 * c_ai) + (2 * c_opp) + (3 * s_count)

    if strategy == "Defensive":
        #Extra penalty if an opponent is about to win; skip cards are very valuable
        if state.opponent1_cards <= 1 or state.opponent2_cards <= 1:
            score -= 20
        score += (2 * s_count)
    elif strategy == "Offensive":
        #Big bonus if AI is about to win; penalise holding too many cards
        if c_ai <= 1:
            score += 20
        score -= (2 * c_ai)

    return score

#STATE TRANSITION
def simulate_move(state, move, player_turn):
    new_ai_cards        = state.ai_cards.copy()
    new_opponent1_cards = state.opponent1_cards
    new_opponent2_cards = state.opponent2_cards
    new_top_card        = state.top_card
    new_deck            = state.deck.copy() if isinstance(state.deck, list) else []

    if player_turn == 0:   # AI's own turn
        if move == "Draw":
            if new_deck:
                new_ai_cards.append(new_deck.pop(0))
        else:
            for c in new_ai_cards:
                if c.color == move.color and c.value == move.value:
                    new_ai_cards.remove(c)
                    new_top_card = move
                    break
    elif player_turn == 1:
        if move == "Draw":
            if new_deck:
                new_deck.pop(0)
                new_opponent1_cards += 1
        else:
            new_opponent1_cards -= 1
            new_top_card = move
    elif player_turn == 2:
        if move == "Draw":
            if new_deck:
                new_deck.pop(0)
                new_opponent2_cards += 1
        else:
            new_opponent2_cards -= 1
            new_top_card = move

    return GameState(new_ai_cards, new_opponent1_cards, new_opponent2_cards, new_top_card, new_deck)

print('Core classes and functions loaded.')

Core classes and functions loaded.


## Evaluation Function Explanation

**Formula:** `Score = 50 - 5×C_AI + 2×C_Opp + 3×S`

| Variable | Meaning | Goal |
|---|---|---|
| C_AI | Cards in our hand | Minimise (0 = win) |
| C_Opp | Average opponent hand size | Maximise |
| S | Skip cards in our hand | Maximise (control) |

**Base score** starts at 50. Every card we hold costs 5 points; every card an opponent holds gives 2 points; each Skip card gives 3 points.

**Defensive tuning (Player 1 — Minimax):**  
- `-20` panic penalty if any opponent has ≤1 card (stop them from winning first)  
- Extra `+2` per Skip card — hoarding Skips to block the opponent about to win

**Offensive tuning (Player 2 — Expectimax):**  
- `+20` bonus when we are 1 card away from winning (race to zero)  
- Extra `-2` per card we hold (extra pressure to shed cards fast)

In [8]:

def minimax(state, depth, alpha, beta, player_turn):
    if depth == 0 or len(state.ai_cards) == 0 or state.opponent1_cards == 0 or state.opponent2_cards == 0:
        return evaluate_state(state, "Defensive"), None

    if player_turn == 0:   # MAX node
        max_eval  = float('-inf')
        best_move = None
        valid_moves = get_valid_moves(state.ai_cards, state.top_card) or ["Draw"]

        for move in valid_moves:
            new_state = simulate_move(state, move, player_turn)
            eval_score, _ = minimax(new_state, depth - 1, alpha, beta, 1)
            if eval_score > max_eval:
                max_eval  = eval_score
                best_move = move
            alpha = max(alpha, eval_score)
            if beta <= alpha:
                break
        return max_eval, best_move

    else:   # MIN node (opponents)
        min_eval      = float('inf')
        possible_moves = [state.top_card, "Draw"]

        for move in possible_moves:
            new_state  = simulate_move(state, move, player_turn)
            next_turn  = 2 if player_turn == 1 else 0
            eval_score, _ = minimax(new_state, depth - 1, alpha, beta, next_turn)
            if eval_score < min_eval:
                min_eval = eval_score
            beta = min(beta, eval_score)
            if beta <= alpha:
                break
        return min_eval, None


def expectimax(state, depth, player_turn):
    if depth == 0 or len(state.ai_cards) == 0 or state.opponent1_cards == 0 or state.opponent2_cards == 0:
        return evaluate_state(state, "Offensive"), None

    if player_turn == 0:   # MAX node
        max_eval  = float('-inf')
        best_move = None
        valid_moves = get_valid_moves(state.ai_cards, state.top_card) or ["Draw"]

        for move in valid_moves:
            new_state = simulate_move(state, move, player_turn)
            eval_score, _ = expectimax(new_state, depth - 1, 1)
            if eval_score > max_eval:
                max_eval  = eval_score
                best_move = move
        return max_eval, best_move

    else:   
        expected_eval  = 0
        possible_moves = [state.top_card, "Draw"]

        for move in possible_moves:
            if move == "Draw" and isinstance(state.deck, list) and state.deck:
                # CHANCE NODE: weight each possible drawn card by its probability
                deck_len     = len(state.deck)
                draw_expected = 0
                for drawn_card in state.deck:
                    probability = 1.0 / deck_len
                    temp_deck   = [c for c in state.deck if c != drawn_card]
                    if drawn_card.matches(state.top_card):
                        temp_state = (GameState(state.ai_cards, state.opponent1_cards + 1,
                                               state.opponent2_cards, drawn_card, temp_deck)
                                      if player_turn == 1 else
                                      GameState(state.ai_cards, state.opponent1_cards,
                                               state.opponent2_cards + 1, drawn_card, temp_deck))
                    else:
                        temp_state = (GameState(state.ai_cards, state.opponent1_cards + 1,
                                               state.opponent2_cards, state.top_card, temp_deck)
                                      if player_turn == 1 else
                                      GameState(state.ai_cards, state.opponent1_cards,
                                               state.opponent2_cards + 1, state.top_card, temp_deck))
                    next_turn   = 2 if player_turn == 1 else 0
                    eval_score, _ = expectimax(temp_state, depth - 1, next_turn)
                    draw_expected += probability * eval_score
                expected_eval += draw_expected
            else:
                new_state  = simulate_move(state, move, player_turn)
                next_turn  = 2 if player_turn == 1 else 0
                eval_score, _ = expectimax(new_state, depth - 1, next_turn)
                expected_eval += eval_score

        expected_eval /= len(possible_moves)
        return expected_eval, None

print('Minimax and Expectimax loaded.')

Minimax and Expectimax loaded.


In [9]:

def print_game_tree(state, depth=3, indent="", player_turn=0):
    if depth == 0:
        score = evaluate_state(state, "Defensive")
        print(indent + "Eval: " + str(round(score, 2)))
        return

    moves  = get_valid_moves(state.ai_cards, state.top_card) or ["Draw"]
    labels = {3: "P1 (MAX - Minimax)", 2: "P2 (Opponent)", 1: "P3 (MIN)"}

    if indent == "":
        print(labels.get(depth, "Terminal"))

    for i, move in enumerate(moves):
        is_last  = (i == len(moves) - 1)
        symbol   = "└" if is_last else "├"
        move_str = move if move == "Draw" else str(move)
        print(indent + symbol + "-- " + move_str)

        new_state    = simulate_move(state, move, 0)
        next_indent  = indent + ("  " if is_last else "| ")
        next_depth   = depth - 1
        next_player  = (player_turn + 1) % 3

        if next_depth > 0:
            next_label   = "CHANCE" if move == "Draw" else labels.get(next_depth, "")
            if next_label:
                print(next_indent + "- " + next_label)
                deeper_indent = next_indent + "  "
            else:
                deeper_indent = next_indent
        else:
            deeper_indent = next_indent

        print_game_tree(new_state, next_depth, deeper_indent, next_player)


def print_decisions_with_scores(state, algorithm="minimax"):
    print("\n" + "-" * 70)
    print("AI DECISION (All possible decisions considered at depth 1):")
    print("-" * 70)

    moves       = get_valid_moves(state.ai_cards, state.top_card) or ["Draw"]
    move_scores = []

    if algorithm == "minimax":
        for move in moves:
            new_state = simulate_move(state, move, 0)
            score, _  = minimax(new_state, 2, float('-inf'), float('inf'), 1)
            move_scores.append((move, score))
    else:
        for move in moves:
            new_state = simulate_move(state, move, 0)
            score, _  = expectimax(new_state, 2, 1)
            move_scores.append((move, score))

    for move, score in sorted(move_scores, key=lambda x: x[1], reverse=True):
        move_str = str(move) if move != "Draw" else "Draw"
        print(f"  Play: {move_str:<20} Expected score: {score:.2f}")

    print("-" * 70 + "\n")

print('Tree printer and decision display loaded.')

Tree printer and decision display loaded.


In [10]:
# ===== COLOR SCHEME =====
COLOR_SCHEME = {
    'Red':     '#E74C3C',
    'Blue':    '#3498DB',
    'Green':   '#2ECC71',
    'Yellow':  '#F39C12',
    'bg_dark': '#1a1a2e',
    'bg_mid':  '#16213e',
    'accent':  '#0F3460',
    'active':  '#e94560',
}
PLAYER_COLORS = ['#E74C3C', '#3498DB', '#2ECC71']

def get_card_color(color_name):
    return COLOR_SCHEME.get(color_name, '#95A5A6')

# ===== MODE SELECTION DIALOG =====
def show_mode_selection():
    root = tk.Tk()
    root.withdraw()
    root.lift()
    root.attributes('-topmost', True)
    result = messagebox.askyesno(
        "Player 3 Mode Selection",
        "Select Player 3 Mode:\n\nYES = Manual (You Play)\nNO = Simulation (All AI)"
    )
    root.destroy()
    return "manual" if result else "simulation"

print('Color scheme and mode selection loaded.')

Color scheme and mode selection loaded.


In [11]:

def play_uno_gui(p3_mode="simulation"):
    print("=" * 70)
    print("Starting UNO AI Game with GUI")
    print("=" * 70)

    # --- Game state ---
    deck           = generate_deck()
    p1_hand        = [deck.pop() for _ in range(5)]
    p2_hand        = [deck.pop() for _ in range(5)]
    p3_hand        = [deck.pop() for _ in range(5)]
    top_card       = deck.pop()
    current_player = 0
    game_running   = True


    gui_queue = queue.Queue()

    # --- Manual mode sync ---
    manual_event  = threading.Event()
    manual_result = [None]

    #BUILD GUI
    root = tk.Tk()
    root.title("UNO AI Championship")
    root.geometry("960x780")
    root.configure(bg=COLOR_SCHEME['bg_dark'])
    root.resizable(False, False)
    root.lift()
    root.attributes('-topmost', True)
    root.after_idle(root.attributes, '-topmost', False)

    #Header
    header_frame = tk.Frame(root, bg=COLOR_SCHEME['accent'], height=70)
    header_frame.pack(fill=tk.X)
    tk.Label(header_frame, text="UNO AI Championship",
             font=("Arial", 22, "bold"),
             bg=COLOR_SCHEME['accent'], fg="white").pack(pady=15)

    #Center area
    center_frame = tk.Frame(root, bg=COLOR_SCHEME['bg_dark'])
    center_frame.pack(pady=10)

    current_player_lbl = tk.Label(center_frame, text="Current Turn: Player 1",
                                  font=("Arial", 13, "bold"),
                                  bg=COLOR_SCHEME['bg_dark'], fg=PLAYER_COLORS[0])
    current_player_lbl.pack(pady=6)

    card_row = tk.Frame(center_frame, bg=COLOR_SCHEME['bg_dark'])
    card_row.pack()

    #Deck box
    deck_box = tk.Frame(card_row, bg='#2c2c54', width=110, height=150,
                        borderwidth=3, relief=tk.RIDGE)
    deck_box.pack(side=tk.LEFT, padx=20)
    deck_box.pack_propagate(False)
    tk.Label(deck_box, text="DECK", font=("Arial", 12, "bold"),
             bg='#2c2c54', fg='#aaa').pack(expand=True)
    deck_count_lbl = tk.Label(deck_box, text=f"{len(deck)}",
                              font=("Arial", 11), bg='#2c2c54', fg='white')
    deck_count_lbl.pack(pady=4)

    #Top card box
    top_card_box = tk.Frame(card_row, bg=get_card_color(top_card.color),
                            width=110, height=150, borderwidth=3, relief=tk.RIDGE)
    top_card_box.pack(side=tk.LEFT, padx=20)
    top_card_box.pack_propagate(False)
    top_card_val_lbl = tk.Label(top_card_box, text=str(top_card.value),
                                font=("Arial", 46, "bold"),
                                bg=get_card_color(top_card.color), fg='white')
    top_card_val_lbl.pack(expand=True)
    top_card_col_lbl = tk.Label(top_card_box, text=top_card.color,
                                font=("Arial", 11, "bold"),
                                bg=get_card_color(top_card.color), fg='white')
    top_card_col_lbl.pack(pady=2)

    #Action banner
    action_frame = tk.Frame(root, bg=COLOR_SCHEME['bg_mid'],
                            borderwidth=2, relief=tk.SUNKEN)
    action_frame.pack(fill=tk.X, padx=10, pady=6)
    action_lbl = tk.Label(action_frame, text="Game starting...",
                          font=("Arial", 12, "italic"),
                          bg=COLOR_SCHEME['bg_mid'], fg="#F39C12", wraplength=900)
    action_lbl.pack(pady=8, padx=10)

    #Player stats boxes
    stats_frame = tk.Frame(root, bg=COLOR_SCHEME['bg_dark'])
    stats_frame.pack(fill=tk.X, padx=15, pady=8)

    player_frames = []
    card_lbls     = []
    player_names_list = ["Player 1", "Player 2", "Player 3"]
    player_strategies = [
        "[Minimax - Defensive]",
        "[Expectimax - Offensive]",
        f"[{'Manual - You' if p3_mode == 'manual' else 'Minimax - AI'}]"
    ]

    for i in range(3):
        pf = tk.Frame(stats_frame, bg=COLOR_SCHEME['bg_mid'],
                      borderwidth=2, relief=tk.RAISED)
        pf.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=5)
        tk.Label(pf, text=f"Player {i+1}",
                 font=("Arial", 11, "bold"),
                 bg=COLOR_SCHEME['bg_mid'], fg=PLAYER_COLORS[i]).pack(pady=5)
        tk.Label(pf, text=player_strategies[i],
                 font=("Arial", 9), bg=COLOR_SCHEME['bg_mid'], fg="#888").pack()
        cl = tk.Label(pf, text="Cards: 5",
                      font=("Arial", 13, "bold"),
                      bg=COLOR_SCHEME['bg_mid'], fg="white")
        cl.pack(pady=8)
        player_frames.append(pf)
        card_lbls.append(cl)

    # P3 hand frame (manual mode)
    p3_hand_outer = tk.Frame(root, bg=COLOR_SCHEME['bg_dark'])
    if p3_mode == "manual":
        p3_hand_outer.pack(fill=tk.X, padx=10, pady=4)
        tk.Label(p3_hand_outer, text="Your Hand - click a card to play it:",
                 font=("Arial", 10, "italic"),
                 bg=COLOR_SCHEME['bg_dark'], fg="#aaa").pack(anchor='w', padx=6)
    p3_hand_frame = tk.Frame(p3_hand_outer, bg=COLOR_SCHEME['bg_dark'])
    p3_hand_frame.pack(fill=tk.X, padx=6)

    # Scrollable log
    log_outer = tk.Frame(root, bg=COLOR_SCHEME['bg_dark'])
    log_outer.pack(fill=tk.BOTH, expand=True, padx=10, pady=6)
    tk.Label(log_outer, text="Game Log:", font=("Arial", 10),
             bg=COLOR_SCHEME['bg_dark'], fg="#888").pack(anchor='w')
    log_text = scrolledtext.ScrolledText(log_outer, height=7,
                                         bg='#0d0d1a', fg='#ccc',
                                         font=("Courier", 9),
                                         state=tk.DISABLED,
                                         borderwidth=0, wrap=tk.WORD)
    log_text.pack(fill=tk.BOTH, expand=True)


    def _do_log(msg):
        """Append a line to the log widget. Must run on main thread."""
        log_text.config(state=tk.NORMAL)
        log_text.insert(tk.END, msg + "\n")
        log_text.see(tk.END)
        log_text.config(state=tk.DISABLED)

    def _do_update_gui(action_text, player_num, done_event):
        """Refresh all widgets. Must run on main thread."""
        if player_num is not None:
            current_player_lbl.config(
                text=f"Current Turn: {player_names_list[player_num]}",
                fg=PLAYER_COLORS[player_num])

        col = get_card_color(top_card.color)
        top_card_box.config(bg=col)
        top_card_val_lbl.config(text=str(top_card.value), bg=col)
        top_card_col_lbl.config(text=top_card.color, bg=col)
        deck_count_lbl.config(text=str(len(deck)))

        for i, hand in enumerate([p1_hand, p2_hand, p3_hand]):
            card_lbls[i].config(text=f"Cards: {len(hand)}")

        for i, frame in enumerate(player_frames):
            is_active = (i == player_num)
            bg = COLOR_SCHEME['active'] if is_active else COLOR_SCHEME['bg_mid']
            frame.config(bg=bg, relief=tk.SUNKEN if is_active else tk.RAISED)
            for child in frame.winfo_children():
                try:
                    child.config(bg=bg)
                except tk.TclError:
                    pass

        action_lbl.config(text=f">  {action_text}")
        done_event.set()

    def _do_show_p3_hand(hand, top_card_ref):
        """Rebuild P3 card buttons. Must run on main thread."""
        for widget in p3_hand_frame.winfo_children():
            widget.destroy()

        valid     = get_valid_moves(hand, top_card_ref)
        valid_set = {(c.color, c.value) for c in valid}

        for card in hand:
            is_valid = (card.color, card.value) in valid_set
            btn = tk.Button(
                p3_hand_frame,
                text=f"{card.color}\n{card.value}",
                font=("Arial", 9, "bold"),
                bg=get_card_color(card.color) if is_valid else '#3a3a4a',
                fg='white', width=7, height=3,
                relief=tk.RAISED,
                state=tk.NORMAL if is_valid else tk.DISABLED,
                command=lambda c=card: _pick(c)
            )
            btn.pack(side=tk.LEFT, padx=3, pady=4)

        draw_btn = tk.Button(
            p3_hand_frame, text="DRAW\nCARD",
            font=("Arial", 9, "bold"),
            bg='#555566', fg='white', width=7, height=3,
            relief=tk.RAISED, command=lambda: _pick("Draw")
        )
        draw_btn.pack(side=tk.LEFT, padx=6, pady=4)

    def _do_clear_p3_hand():
        for widget in p3_hand_frame.winfo_children():
            widget.destroy()


    def process_queue():
        """Drain the gui_queue and execute each task on the main thread."""
        try:
            while True:
                task = gui_queue.get_nowait()
                cmd  = task[0]

                if cmd == 'log':
                    _do_log(task[1])

                elif cmd == 'update':
                    _, action_text, player_num, done_event = task
                    _do_update_gui(action_text, player_num, done_event)

                elif cmd == 'show_p3_hand':
                    _, hand, top_card_ref = task
                    _do_show_p3_hand(hand, top_card_ref)

                elif cmd == 'clear_p3_hand':
                    _do_clear_p3_hand()

                elif cmd == 'destroy':
                    root.destroy()
                    return  

        except queue.Empty:
            pass

        root.after(50, process_queue)


    def log(msg):
        """Queue a log append. Never calls root.after directly."""
        gui_queue.put(('log', msg))

    def update_gui(turn_text, action_text, player_num=None):
        """Queue a full GUI refresh and block until it completes."""
        done = threading.Event()
        gui_queue.put(('update', action_text, player_num, done))
        done.wait()          # wait for main thread to finish the update
        time.sleep(1.5)      # safe: we're on the background game thread

    def show_p3_hand(hand, top_card_ref):
        gui_queue.put(('show_p3_hand', hand, top_card_ref))

    def clear_p3_hand():
        gui_queue.put(('clear_p3_hand',))

    def _pick(move):
        """Called when P3 clicks a card button (main thread)."""
        manual_result[0] = move
        manual_event.set()

    def get_manual_move(hand, top_card_ref):
        """Block game thread until player clicks a card."""
        manual_event.clear()
        manual_result[0] = None
        show_p3_hand(hand, top_card_ref)
        manual_event.wait()
        clear_p3_hand()
        return manual_result[0]


    def game_loop():
        nonlocal current_player, top_card, game_running

        try:
            while game_running:

                # Check win / end conditions
                if len(p1_hand) == 0:
                    update_gui("Game Over!", "Player 1 Wins!", 0)
                    log(">>> GAME OVER - Player 1 Wins!")
                    print("Player 1 Wins")
                    break
                if len(p2_hand) == 0:
                    update_gui("Game Over!", "Player 2 Wins!", 1)
                    log(">>> GAME OVER - Player 2 Wins!")
                    print("Player 2 Wins")
                    break
                if len(p3_hand) == 0:
                    update_gui("Game Over!", "Player 3 Wins!", 2)
                    log(">>> GAME OVER - Player 3 Wins!")
                    print("Player 3 Wins")
                    break
                if not deck:
                    update_gui("Game Over!", "Deck empty - Draw!", None)
                    log(">>> GAME OVER - Deck Empty!")
                    print("Deck Empty")
                    break

                
                if current_player == 0:
                    state = GameState(p1_hand, len(p2_hand), len(p3_hand), top_card, deck)
                    print("\n" + "=" * 70)
                    print("PLAYER 1 (Minimax - Defensive) TURN")
                    print("=" * 70)
                    print(f"Top Card: {top_card}")
                    print(f"P1 Hand: {p1_hand}")
                    print("\nGame Tree (depth 2):")
                    print_game_tree(state, 2)
                    print_decisions_with_scores(state, "minimax")

                    score, move = minimax(state, 3, float('-inf'), float('inf'), 0)
                    print(f"BEST MOVE: {move} (Score: {score:.2f})")
                    log(f"[P1 Minimax] Top={top_card} | Move: {move} | Score: {score:.2f}")

                    if move == "Draw" or move is None:
                        if deck:
                            p1_hand.append(deck.pop())
                        update_gui("P1 Turn", "P1 draws a card", 0)
                        current_player = 1
                    else:
                        p1_hand[:] = [c for c in p1_hand
                                      if not (c.color == move.color and c.value == move.value)]
                        top_card = move
                        if top_card.value == 'Skip':
                            update_gui("P1 Turn", f"P1 plays {move} - P2 SKIPPED!", 0)
                            current_player = 2
                        else:
                            update_gui("P1 Turn", f"P1 plays {move}", 0)
                            current_player = 1

               
                elif current_player == 1:
                    state = GameState(p2_hand, len(p3_hand), len(p1_hand), top_card, deck)
                    print("\n" + "=" * 70)
                    print("PLAYER 2 (Expectimax - Offensive) TURN")
                    print("=" * 70)
                    print(f"Top Card: {top_card}")
                    print(f"P2 Hand: {p2_hand}")
                    print("\nGame Tree (depth 2):")
                    print_game_tree(state, 2)
                    print_decisions_with_scores(state, "expectimax")

                    score, move = expectimax(state, 3, 0)
                    print(f"BEST MOVE: {move} (Score: {score:.2f})")
                    log(f"[P2 Expectimax] Top={top_card} | Move: {move} | Score: {score:.2f}")

                    if move == "Draw" or move is None:
                        if deck:
                            p2_hand.append(deck.pop())
                        update_gui("P2 Turn", "P2 draws a card", 1)
                        current_player = 2
                    else:
                        p2_hand[:] = [c for c in p2_hand
                                      if not (c.color == move.color and c.value == move.value)]
                        top_card = move
                        if top_card.value == 'Skip':
                            update_gui("P2 Turn", f"P2 plays {move} - P3 SKIPPED!", 1)
                            current_player = 0
                        else:
                            update_gui("P2 Turn", f"P2 plays {move}", 1)
                            current_player = 2

             
                elif current_player == 2:

                    if p3_mode == "manual":
                        log(f"[P3 Manual] Your turn! Top card: {top_card}")
                        update_gui("P3 Turn", "Your turn! Click a card or DRAW", 2)

                        move = get_manual_move(p3_hand, top_card)

                        if move == "Draw":
                            if deck:
                                drawn = deck.pop()
                                p3_hand.append(drawn)
                                log(f"[P3 Manual] You drew: {drawn}")
                                print(f"You drew: {drawn}")
                            update_gui("P3 Turn", "You drew a card", 2)
                            current_player = 0
                        else:
                            p3_hand[:] = [c for c in p3_hand
                                          if not (c.color == move.color and c.value == move.value)]
                            top_card = move
                            log(f"[P3 Manual] You played: {move}")
                            print(f"You played: {move}")
                            if top_card.value == 'Skip':
                                update_gui("P3 Turn", f"You played {move} - P1 SKIPPED!", 2)
                                current_player = 1
                            else:
                                update_gui("P3 Turn", f"You played {move}", 2)
                                current_player = 0

                    else:  
                        state = GameState(p3_hand, len(p1_hand), len(p2_hand), top_card, deck)
                        print("\n" + "=" * 70)
                        print("PLAYER 3 (Minimax - Adaptive AI) TURN")
                        print("=" * 70)
                        print(f"Top Card: {top_card}")
                        print(f"P3 Hand: {p3_hand}")
                        print("\nGame Tree (depth 2):")
                        print_game_tree(state, 2)
                        print_decisions_with_scores(state, "minimax")

                        score, move = minimax(state, 3, float('-inf'), float('inf'), 0)
                        print(f"BEST MOVE: {move} (Score: {score:.2f})")
                        log(f"[P3 Minimax] Top={top_card} | Move: {move} | Score: {score:.2f}")

                        if move == "Draw" or move is None:
                            if deck:
                                p3_hand.append(deck.pop())
                            update_gui("P3 Turn", "P3 draws a card", 2)
                            current_player = 0
                        else:
                            p3_hand[:] = [c for c in p3_hand
                                          if not (c.color == move.color and c.value == move.value)]
                            top_card = move
                            if top_card.value == 'Skip':
                                update_gui("P3 Turn", f"P3 plays {move} - P1 SKIPPED!", 2)
                                current_player = 1
                            else:
                                update_gui("P3 Turn", f"P3 plays {move}", 2)
                                current_player = 0

                print(f"Cards - P1:{len(p1_hand)}  P2:{len(p2_hand)}  P3:{len(p3_hand)}  Deck:{len(deck)}")
                print("-" * 70)

        except Exception as e:
            print(f"\n[ERROR in game_loop]: {e}")
            import traceback
            traceback.print_exc()
            log(f"[ERROR]: {e}")
        finally:
            time.sleep(4)
            gui_queue.put(('destroy',))

    root.after(50, process_queue)
    game_thread = threading.Thread(target=game_loop, daemon=True)
    root.after(600, game_thread.start)
    root.mainloop()

print('GUI game function loaded successfully!')

GUI game function loaded successfully!


## Game Tree Explanation

The game tree has **3 levels** corresponding to each player's turn:

```
                    P1 (MAX node - Minimax)
                   /         |          \
              move1        move2         Draw
                |            |             |
           P2 (MIN)      P2 (MIN)       CHANCE NODE
           /    \        /    \        (weighted by
         play  draw    play  draw       deck probability)
           |    |        |    |
        P3(MIN) ...   P3(MIN) ...
           |               |
         Eval            Eval
```

**Node types:**
- **MAX node (depth 3)** — P1's turn. AI picks the move with the highest evaluation score.
- **MIN node (depth 2, 1)** — Opponents (P2, P3) minimise the AI's score (worst-case assumption).
- **CHANCE node** — Triggered when the Draw action is chosen in Expectimax. Each card in the deck has probability `1 / deck_size`. The expected value is `Σ P(card) × score(child_state)`.

**Alpha-Beta Pruning** in Minimax: branches are cut when `beta ≤ alpha`, avoiding exploration of moves that can never affect the final decision.

In [12]:
mode = show_mode_selection()
play_uno_gui(p3_mode=mode)

Starting UNO AI Game with GUI

PLAYER 1 (Minimax - Defensive) TURN
Top Card: Blue 2
P1 Hand: [Blue 7, Green 7, Red 8, Blue 6, Yellow 5]

Game Tree (depth 2):
P2 (Opponent)
├-- Blue 7
| - P3 (MIN)
|   ├-- Green 7
|   | Eval: 45.0
|   └-- Blue 6
|     Eval: 45.0
└-- Blue 6
  - P3 (MIN)
    └-- Blue 7
      Eval: 45.0

----------------------------------------------------------------------
AI DECISION (All possible decisions considered at depth 1):
----------------------------------------------------------------------
  Play: Blue 7               Expected score: 38.00
  Play: Blue 6               Expected score: 38.00
----------------------------------------------------------------------

BEST MOVE: Blue 7 (Score: 38.00)


Cards - P1:4  P2:5  P3:5  Deck:28
----------------------------------------------------------------------

PLAYER 2 (Expectimax - Offensive) TURN
Top Card: Blue 7
P2 Hand: [Green 9, Green Skip, Green 2, Blue 0, Green 1]

Game Tree (depth 2):
P2 (Opponent)
└-- Blue 0
  - P3 (MIN)
    └-- Draw
      Eval: 39.0

----------------------------------------------------------------------
AI DECISION (All possible decisions considered at depth 1):
----------------------------------------------------------------------
  Play: Blue 0               Expected score: 34.00
----------------------------------------------------------------------

BEST MOVE: Blue 0 (Score: 34.00)


## Algorithm Comparison

### Strategy Employed

| | Player 1 — Minimax | Player 2 — Expectimax |
|---|---|---|
| **Algorithm** | Minimax with Alpha-Beta Pruning | Expectimax |
| **Strategy** | Defensive | Offensive |
| **Core goal** | Avoid losing; block opponents | Win as fast as possible |
| **Skip cards** | Hoarded to block opponents near 0 | Played aggressively to shed cards |
| **Draw action** | Treated as a definite bad move (MIN) | Treated as a chance node with probability |

### Which Algorithm Performed Best?

In simulation runs, **Expectimax (Player 2 — Offensive)** tends to win more often in short games because it races to shed cards, while Minimax (Defensive) can stall by holding Skip cards.

However, **Minimax performs better in longer games** where opponents are frequently close to winning — the `-20` penalty kicks in and forces P1 to play Skips at exactly the right moment to reset opponent hands.

### Why Is Expectimax Superior for Card Games?

1. **Handles uncertainty** — The Draw action does not have a deterministic outcome. Expectimax models this correctly as a chance node: `E[score] = Σ P(drawn_card) × score(state_after_draw)`. Minimax incorrectly assumes the opponent always draws the worst possible card.

2. **More realistic opponent model** — In a card game with partial information, opponents do not always play optimally against us. Expectimax's random/probabilistic opponent model is closer to reality than Minimax's worst-case assumption.

3. **Offensive strategy aligns with the goal** — UNO is won by emptying your hand. An algorithm that maximises card-shedding speed outperforms one that focuses on blocking.

### Why Does Minimax Still Have Value?

Minimax's worst-case guarantee is valuable when opponents are strategic. If P2 and P3 were also running Expectimax and actively targeting P1, the Defensive Minimax would be more robust because it never assumes opponents will make suboptimal moves.